# 🚀 Quickstart Tutorial

Welcome to the **Fair Synthetic Data Generator**! This tutorial will get you
started with generating fair synthetic data in just a few minutes.

## What You'll Learn

1. **Setup and Installation**
2. **Loading and Preparing Data**
3. **Training a Fair Generator**
4. **Generating Synthetic Data**
5. **Evaluating Results**

## 1. Setup and Installation

First, let's set up our environment.

In [ ]:
# Install the package (if not already installed)
# !pip install fair-synthetic-generator

# Or install from source
# !pip install -e .

print("✅ Package ready!")

In [ ]:
# Import required libraries
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd().parent.parent
sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Main imports
from src.synthesis import GeneratorPipeline
from src.evaluation import FairnessReport, FidelityEvaluator
from src.configs import Config

# Suppress warnings
import warnings
warnings.filterwarnings('ignore')

print("📦 All imports successful!")

## 2. Loading and Preparing Data

For this tutorial, we'll create a sample dataset. In practice, you would load your own data.

In [ ]:
# Create a sample dataset
np.random.seed(42)
n_samples = 5000

data = pd.DataFrame({
    'age': np.random.randint(18, 70, n_samples),
    'gender': np.random.choice(['Male', 'Female'], n_samples),
    'income': np.random.exponential(50000, n_samples) + 20000,
    'education_years': np.random.randint(8, 20, n_samples),
    'credit_score': np.random.normal(700, 50, n_samples).clip(300, 850),
    'approved': np.random.choice([0, 1], n_samples, p=[0.7, 0.3])
})

# Introduce some bias (important: we want to remove this!)
data.loc[data['gender'] == 'Female', 'approved'] = np.random.choice(
    [0, 1], sum(data['gender'] == 'Female'), p=[0.75, 0.25]
)

print(f"📊 Dataset shape: {data.shape}")
print(f"\nFirst few rows:")
data.head()

In [ ]:
# Define sensitive attributes
SENSITIVE_ATTRIBUTES = ['gender']
TARGET_COLUMN = 'approved'

# Check current bias
print("\n🔍 Current bias in data:")
print(data.groupby('gender')[TARGET_COLUMN].mean())

bias_before = data.groupby('gender')[TARGET_COLUMN].mean().diff().dropna().values[0]
print(f"\nBias (difference in approval rates): {bias_before:.2%}")

## 3. Training a Fair Generator

Now let's train a generator that produces fair synthetic data.

In [ ]:
# Configuration
config = Config({
    'model': {
        'type': 'vae',
        'latent_dim': 32,
        'hidden_dims': [128, 256, 128]
    },
    'fairness': {
        'enabled': True,
        'type': 'adversarial',
        'sensitive_attributes': SENSITIVE_ATTRIBUTES,
        'lambda_fairness': 0.5
    },
    'training': {
        'epochs': 50,
        'batch_size': 256,
        'learning_rate': 1e-3
    }
})

print("📋 Configuration:")
print(config)

In [ ]:
# Initialize the pipeline
pipeline = GeneratorPipeline(config)

print("✅ Pipeline initialized")

In [ ]:
# Train the generator
print("🏋️ Training fair generator...")
print("=" * 50)

pipeline.fit(data, sensitive_attributes=SENSITIVE_ATTRIBUTES)

print("\n✅ Training complete!")

## 4. Generating Synthetic Data

Now let's generate synthetic data using our trained fair generator.

In [ ]:
# Generate synthetic data
n_synthetic = 3000

synthetic_data = pipeline.generate(n_synthetic)

print(f"🎨 Generated {len(synthetic_data)} synthetic samples")
print(f"\nSynthetic data preview:")
synthetic_data.head()

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

numeric_cols = ['age', 'income', 'education_years', 'credit_score']

for idx, col in enumerate(numeric_cols):
    ax = axes.flatten()[idx]
    ax.hist(data[col], bins=30, alpha=0.5, label='Real', density=True)
    ax.hist(synthetic_data[col], bins=30, alpha=0.5, label='Synthetic', density=True)
    ax.set_title(col)
    ax.legend()

# Gender distribution
ax = axes[1, 2]
real_gender = data['gender'].value_counts(normalize=True)
synth_gender = synthetic_data['gender'].value_counts(normalize=True)

x = np.arange(2)
width = 0.35
ax.bar(x - width/2, [real_gender.get('Male', 0), real_gender.get('Female', 0)],
       width, label='Real', color='steelblue')
ax.bar(x + width/2, [synth_gender.get('Male', 0), synth_gender.get('Female', 0)],
       width, label='Synthetic', color='coral')
ax.set_xticks(x)
ax.set_xticklabels(['Male', 'Female'])
ax.set_title('Gender Distribution')
ax.legend()

plt.suptitle('Real vs Synthetic Data Comparison', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Evaluating Results

Let's evaluate both fidelity (how similar) and fairness (how fair) of our synthetic data.

In [ ]:
# Evaluate fairness
print("⚖️ FAIRNESS EVALUATION")
print("=" * 50)

fairness_report = FairnessReport(
    sensitive_attributes=SENSITIVE_ATTRIBUTES,
    target_column=TARGET_COLUMN
)

report = fairness_report.evaluate(
    real_data=data,
    synthetic_data=synthetic_data
)

print("\n📊 Fairness Metrics:")
for metric, value in report['fairness_metrics'].items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Evaluate fidelity
print("\n📊 FIDELITY EVALUATION")
print("=" * 50)

fidelity_evaluator = FidelityEvaluator()
fidelity_scores = fidelity_evaluator.evaluate(data, synthetic_data)

print("\nFidelity Metrics:")
for metric, value in fidelity_scores.items():
    print(f"  {metric}: {value:.4f}")

In [ ]:
# Summary comparison
print("\n" + "=" * 60)
print("📈 SUMMARY")
print("=" * 60)

print(f"\n📊 Real Data Bias:")
print(f"   Approval rate difference by gender: {bias_before:.2%}")

print(f"\n⚖️ Synthetic Data Fairness:")
print(f"   Demographic parity difference: {report['fairness_metrics']['demographic_parity_difference']:.4f}")

print(f"\n🎯 Fidelity Score: {fidelity_scores['overall_fidelity']:.4f}")

print("\n✅ Quickstart complete!")

## Next Steps

Congratulations! You've successfully:

✅ Set up the environment  
✅ Created and prepared a dataset  
✅ Trained a fair generator  
✅ Generated synthetic data  
✅ Evaluated fairness and fidelity  

### What to try next:

1. **Custom Fairness Constraints**: Learn to define your own fairness constraints  
2. **Multi-modal Synthesis**: Work with text and image data  
3. **Advanced Models**: Try different generator architectures (GAN, Diffusion)  
4. **API Usage**: Integrate with your applications via REST API